# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
%load_ext dotenv
%dotenv


In [2]:
import dask.dataframe as dd

c:\Users\onema\miniconda3\envs\dsi_participant\lib\site-packages\dask\dataframe\_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 11.0.0. Please consider upgrading.
  warnings.warn(


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [4]:
import os
from glob import glob

# Write your code below.
PRICE_DATA = os.getenv("PRICE_DATA")
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [10]:
# Write your code below.
dd_px = dd.read_parquet(parquet_files).set_index("ticker")
dd_feat = dd_px.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(Close_lag_1 = x['Close'].shift(1), 
                       Adj_Close_lag_1 = x['Adj Close'].shift(1),
                       Returns = x['Close'] / x['Close'].shift(1) - 1,
                       Hi_Lo_Range = x['High'] - x['Low'], 
                       )
)

dd_feat.head()

C:\Users\onema\AppData\Local\Temp\ipykernel_6744\1822710022.py:3: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_feat = dd_px.groupby('ticker', group_keys=False).apply(


,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,Returns,Hi_Lo_Range
ticker,,,,,,,,,,,,,
ACN,2016-07-05,113.510002,113.559998,112.629997,113.099998,106.026108,2041100.0,ACN.csv,2016,NaN,NaN,NaN,0.930000
ACN,2016-07-06,112.629997,113.519997,111.959999,113.519997,106.419830,2667200.0,ACN.csv,2016,113.099998,106.026108,0.003714,1.559998
ACN,2016-07-07,113.510002,113.900002,112.510002,112.699997,105.651115,2050500.0,ACN.csv,2016,113.519997,106.419830,-0.007223,1.389999
ACN,2016-07-08,113.730003,115.239998,113.169998,115.110001,107.910400,1962800.0,ACN.csv,2016,112.699997,105.651115,0.021384,2.070000
ACN,2016-07-11,115.699997,115.980003,115.129997,115.459999,108.238510,1522300.0,ACN.csv,2016,115.110001,107.910400,0.003041,0.850006


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [13]:
# Write your code below.

pd_feat = dd_feat.compute()
pd_feat = pd_feat.sort_values(["ticker"])

pd_feat["returns_rollling10"] = (
    pd_feat.groupby("ticker")["Returns"]
    .transform(lambda x: x.rolling(10).mean())
)

pd_feat.head(100)

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,Returns,Hi_Lo_Range,returns_rollling10
ticker,,,,,,,,,,,,,,
ACN,2016-07-05,113.510002,113.559998,112.629997,113.099998,106.026108,2041100.0,ACN.csv,2016,NaN,NaN,NaN,0.930000,NaN
ACN,2010-04-16,42.950001,43.360001,42.950001,43.150002,35.045681,4399800.0,ACN.csv,2010,43.240002,35.118782,-0.002081,0.410000,NaN
ACN,2010-04-15,43.040001,43.320000,42.980000,43.240002,35.118782,3651900.0,ACN.csv,2010,43.040001,34.956333,0.004647,0.340000,NaN
ACN,2010-04-14,43.160000,43.360001,42.880001,43.040001,34.956333,4426100.0,ACN.csv,2010,43.099998,34.700489,-0.001392,0.480000,NaN
ACN,2010-04-13,42.830002,43.119999,42.549999,43.099998,34.700489,4201500.0,ACN.csv,2010,43.070000,34.676342,0.000697,0.570000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ACN,2010-08-23,38.689999,38.730000,37.509998,38.000000,30.862934,5212400.0,ACN.csv,2010,38.419998,31.204052,-0.010932,1.220001,-0.004053
ACN,2010-08-20,38.709999,38.759998,38.189999,38.419998,31.204052,4549300.0,ACN.csv,2010,38.900002,31.593897,-0.012339,0.570000,-0.004914
ACN,2010-08-19,39.419998,39.619999,38.650002,38.900002,31.593897,4763400.0,ACN.csv,2010,39.660000,32.211159,-0.019163,0.969997,-0.008139


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
No, Dask has its own tool for calculating moving averages.

+ Would it have been better to do it in Dask? Why?
Not necessarily considering the size of the data set isn't large enough for its benefit of computing efficiency and scalability.
(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.